In [ ]:
!pip install pandas openpyxl requests

In [ ]:
import time, random, re, html
from getpass import getpass
from getpass import getpass

import requests
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment


In [ ]:
PRODUCT = input("Masukkan hscode product: ").strip()
if not PRODUCT:
    raise ValueError("hscode product jangan dikosongin!!")

safe_name = re.sub(r"[^a-zA-Z0-9_-]+", "_", PRODUCT).strip("_")

OUTPUT_XLSX = f"/content/data1688_{safe_name}_FINAL.xlsx"
OUTPUT_CSV = f"/content/data1688_{safe_name}_FINAL.csv"
CHECKPOINT_CSV = f"/content/data1688_{safe_name}_checkpoint.csv"

print("\n--- AUTHORIZATION DATA1688 ---")

# bersihkan spasi, newline, dan tanda kutip yang ikut ke-paste
AUTHORIZATION = getpass("Paste Auth di sini: ").strip().strip('"').strip("'")
if not AUTHORIZATION:
    raise ValueError("Auth token harus diisi buat nembus API-nya.")
print(f"Auth length: {len(AUTHORIZATION)} | awal: {AUTHORIZATION[:12]}...")

API_URL = "https://dd.data1688.com/stage-v3/es/search/searchGlobal"

BASE_PARAMS = {
    "preciseMatching[0]": "1",
    "datatype": "2",
    "DEST_COUNTRY_TC": "ID",
    "HS_CODE": PRODUCT,
    "date[0]": "2026-01-01",
    "date[1]": "2026-10-16",
    "dataSource": "全球",
    "isInputPrecise": "false",
    "destState[0]": "全部",
    "relationFlag": "0",
    "csRelationFlag": "0",
    "matchingMethod": "1",
    "pageNo": "1",
    "pageSize": "20",
    "tabName": "数据列表",
    "isExportFlag": "false",
    "sort": "query_date",
    "order": "desc",
}

HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Authorization": AUTHORIZATION,
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"),
    "Referer": "https://dd.data1688.com/",
    "Origin": "https://dd.data1688.com",
    "Accept-Language": "en-US,en;q=0.9,id;q=0.8",
}

session = requests.Session()
session.headers.update(HEADERS)

TRANSLATION_MAP = {
    "进口": "Import", "出口": "Export",
    "印度尼西亚": "Indonesia", "澳大利亚": "Australia",
    "巴西": "Brazil", "日本": "Japan", "新西兰": "New Zealand",
    "美国": "United States", "加拿大": "Canada", "印度": "India",
    "中国": "China", "阿根廷": "Argentina", "乌拉圭": "Uruguay",
    "巴拉圭": "Paraguay", "智利": "Chile", "墨西哥": "Mexico",
    "西班牙": "Spain", "法国": "France", "德国": "Germany",
    "英国": "United Kingdom", "意大利": "Italy", "荷兰": "Netherlands",
    "爱尔兰": "Ireland", "丹麦": "Denmark", "比利时": "Belgium",
    "南非": "South Africa",
}


def clean_html(val):
    if not val:
        return None
    val = html.unescape(str(val))
    val = re.sub(r"<[^>]*>", "", val)
    return re.sub(r"\s+", " ", val).strip()

def translate_value(val):
    val = clean_html(val)
    if not val:
        return None
    return TRANSLATION_MAP.get(val, val)

def random_delay():
    jeda = random.uniform(2.0, 4.0)
    print(f"   [+] loading {jeda:.2f} detik...")
    time.sleep(jeda)

def preview(text, n=400):
    return repr(text[:n]) if text else "'' (KOSONG)"

def fetch_page(page_number, max_retries=5):
    params = BASE_PARAMS.copy()
    params["pageNo"] = str(page_number)
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            print(f"-> Tarik halaman {page_number} (Coba {attempt}/{max_retries})")
            resp = session.get(API_URL, params=params, timeout=30)

            if resp.status_code == 429:
                wait_time = 20 * attempt
                print(f"   [!] Kena rate limit, nunggu {wait_time} detik dulu.")
                time.sleep(wait_time)
                last_error = "429 rate limit"
                continue

            if 500 <= resp.status_code <= 599:
                print(f"   [!] Server error {resp.status_code}, retrying...")
                time.sleep(5 * attempt)
                last_error = f"HTTP {resp.status_code}"
                continue

            if resp.status_code in (401, 403):
                raise PermissionError(
                    f"HTTP {resp.status_code}: token salah/expired atau IP diblokir. "
                    f"Body: {preview(resp.text)}"
                )

            resp.raise_for_status()

            try:
                res_json = resp.json()
            except ValueError:
                print(f"   [!] Bukan JSON | Status={resp.status_code} | "
                      f"Type={resp.headers.get('Content-Type', '')}")
                print(f"   [!] URL akhir : {resp.url}")
                print(f"   [!] Body      : {preview(resp.text)}")
                last_error = "response bukan JSON"
                time.sleep(5 * attempt)
                continue

            api_code = res_json.get("code")
            if api_code == 401:
                raise PermissionError("Auth expired atau salah. Ambil ulang dari tab Network.")
            if api_code != 200:
                raise ValueError(f"API Error {api_code}: {res_json.get('msg')}")

            return res_json.get("data", {}) or {}

        except PermissionError:
            raise
        except requests.RequestException as e:
            print(f"   [!] Jaringan ngelag: {e}")
            last_error = str(e)
            time.sleep(5 * attempt)

    raise RuntimeError(f"Gagal narik halaman {page_number} setelah {max_retries} percobaan. Terakhir: {last_error}")

def save_checkpoint(data_list):
    if data_list:
        pd.json_normalize(data_list).to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8-sig")

Masukkan hscode product: 02013000

--- AUTHORIZATION DATA1688 ---
Paste Auth di sini: ··········
Auth length: 281 | awal: Bearer eyJhb...


In [ ]:
r = session.get(API_URL, params=BASE_PARAMS, timeout=30)
j = r.json()
print("code     :", j.get("code"))
print("msg      :", j.get("msg"))
data = j.get("data") or {}
print("totalRow :", data.get("totalRow"))
print("totalPage:", data.get("totalPage"))
print("contoh   :", (data.get("param") or [None])[0])

code     : 200
msg      : None
totalRow : 4001
totalPage: 201
contoh   : {'cargo_value_usd': '4439.8746', 'query_date': '2026-08-28', 'shipper_ref_id': 'AU066164332', 'derived_quantity': '377.82 ( KILOGRAM (KGM) ) ', 'rowkey': 'ID0101015122704271', 'data_type': '进口', 'cargo_description': 'DAGING SAPI SEGAR TANPA TULANG PRS RUMP', 'consignee_name': 'BAYU LESTARI', 'derived_quantity_unit_code': 'KILOGRAM (KGM)', 'hs_code': '<font color=#af17af>02013000</font>', 'destination_country_code': '印度尼西亚', 'data_source_type': '01', 'consignee_ref_id': '01169714e8db4932d2b77ce9fe0c99df', 'data_source': '印度尼西亚', 'shipper_name': 'CAMPBELL FOODS AUSTRALIA PTY LTD', 'derived_weight_kg': '377.82', 'origin_country_code': '澳大利亚', 'id': 'ID0101015122704271', 'index': 'id_01_imp_index_2026_v6', 'isTop': 0, 'isCollection': 0}


In [ ]:
print(f"\n=== MULAI SCRAPING: {PRODUCT.upper()} ===")

all_records = []

print("\nNarik metadata dari halaman 1...")
first_data = fetch_page(1)

total_rows = int(first_data.get("totalRow", 0))
total_pages = int(first_data.get("totalPage", 1))
records = first_data.get("param", []) or []

all_records.extend(records)
save_checkpoint(all_records)

print(f"\nInfo Target:")
print(f"- Total baris data API: {total_rows}")
print(f"- Total halaman     : {total_pages}")
print(f"- Dapet di Hal 1    : {len(records)} baris\n")

for page_no in range(2, total_pages + 1):
    random_delay()
    data = fetch_page(page_no)

    page_records = data.get("param", []) or []
    all_records.extend(page_records)

    save_checkpoint(all_records)
    print(f"   [v] Halaman {page_no} beres. Total dapet sejauh ini: {len(all_records)} baris")

print("\n=== PROSES TARIK DATA SELESAI ===")


=== MULAI SCRAPING: 02013000 ===

Narik metadata dari halaman 1...
-> Tarik halaman 1 (Coba 1/5)

Info Target:
- Total baris data API: 4001
- Total halaman     : 201
- Dapet di Hal 1    : 20 baris

   [+] loading 2.84 detik...
-> Tarik halaman 2 (Coba 1/5)
   [v] Halaman 2 beres. Total dapet sejauh ini: 40 baris
   [+] loading 3.53 detik...
-> Tarik halaman 3 (Coba 1/5)
   [v] Halaman 3 beres. Total dapet sejauh ini: 60 baris
   [+] loading 3.55 detik...
-> Tarik halaman 4 (Coba 1/5)
   [v] Halaman 4 beres. Total dapet sejauh ini: 80 baris
   [+] loading 2.78 detik...
-> Tarik halaman 5 (Coba 1/5)
   [v] Halaman 5 beres. Total dapet sejauh ini: 100 baris
   [+] loading 3.09 detik...
-> Tarik halaman 6 (Coba 1/5)
   [v] Halaman 6 beres. Total dapet sejauh ini: 120 baris
   [+] loading 3.78 detik...
-> Tarik halaman 7 (Coba 1/5)
   [v] Halaman 7 beres. Total dapet sejauh ini: 140 baris
   [+] loading 3.19 detik...
-> Tarik halaman 8 (Coba 1/5)
   [v] Halaman 8 beres. Total dapet sejauh 

In [ ]:
print("Mulai rapihin data mentah ke tabel...")

def parse_qty(val):
    """Ambil angka di awal string, mis. '377.82 ( KILOGRAM (KGM) ) ' -> 377.82"""
    if val is None:
        return None
    m = re.match(r"\s*(-?[\d.,]+)", str(val))
    if not m:
        return None
    return pd.to_numeric(m.group(1).replace(",", ""), errors="coerce")

def format_record(r):
    # Mapping
    ds = translate_value(r.get("data_source"))
    pc = translate_value(r.get("destination_country_code"))

    return {
        "Data source": ds if ds else "Indonesia",
        "Import & Export": translate_value(r.get("data_type")),
        "date": clean_html(r.get("query_date")),
        "HS code": clean_html(r.get("hs_code")),
        "Product DESC": clean_html(r.get("cargo_description")),
        "Buyers": clean_html(r.get("consignee_name")),
        "Supplier": clean_html(r.get("shipper_name")),
        "Weight": pd.to_numeric(r.get("derived_weight_kg"), errors="coerce"),
        "Quantity": parse_qty(r.get("derived_quantity")),
        "Amount": pd.to_numeric(r.get("cargo_value_usd"), errors="coerce"),
        "Purpose Country": pc if pc else "Indonesia",
        "Origin Country": translate_value(r.get("origin_country_code"))
    }

clean_rows = [format_record(r) for r in all_records]
df = pd.DataFrame(clean_rows)

df["HS code"] = df["HS code"].astype("string")

# Cek hasil sebelum ekspor
print(df.head())
print("\nJumlah kolom kosong (NaN):")
print(df.isna().sum())

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

raw_df = pd.json_normalize(all_records)
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Data", index=False)
    raw_df.to_excel(writer, sheet_name="Raw API", index=False)

wb = load_workbook(OUTPUT_XLSX)
ws = wb["Data"]
ws.freeze_panes = "A2"
ws.auto_filter.ref = ws.dimensions

for cell in ws[1]:
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal="center", vertical="center")

col_widths = {"A":18, "B":18, "C":15, "D":15, "E":60, "F":40, "G":40, "H":16, "I":16, "J":18, "K":22, "L":22}
for col, width in col_widths.items():
    ws.column_dimensions[col].width = width

for row in ws.iter_rows(min_row=2):
    for i in [4, 5, 6]:
        row[i].alignment = Alignment(vertical="top", wrap_text=True)

for cell in ws["D"][1:]:
    cell.number_format = "@"

wb.save(OUTPUT_XLSX)
print("File Excel udah berhasil dibuat dan diformat!")

Mulai rapihin data mentah ke tabel...
  Data source Import & Export        date   HS code  \
0   Indonesia          Import  2026-08-28  02013000   
1   Indonesia          Import  2026-08-28  02013000   
2   Indonesia          Import  2026-08-28  02013000   
3   Indonesia          Import  2026-08-28  02013000   
4   Indonesia          Import  2026-08-28  02013000   

                                        Product DESC  \
0            DAGING SAPI SEGAR TANPA TULANG PRS RUMP   
1       DAGING SAPI SEGAR TANPA TULANG, A TENDERLOIN   
2      DAGING SAPI SEGAR TANPA TULANG, PRS CUBE ROLL   
3   *YP-PEBO* POINT END BRISKET DECKLE OFF GRAIN FED   
4  *YP-TDRSO* TENDERLOIN SIDE STRAP OFF GRAIN FED...   

                         Buyers                          Supplier  Weight  \
0                  BAYU LESTARI  CAMPBELL FOODS AUSTRALIA PTY LTD  377.82   
1                  BAYU LESTARI  CAMPBELL FOODS AUSTRALIA PTY LTD  141.91   
2                  BAYU LESTARI  CAMPBELL FOODS AUSTRALIA PTY L

In [ ]:
print("\n--- VALIDASI AKHIR ---")
print(f"Target jumlah data dari web : {total_rows}")
print(f"Total yang berhasil discrap : {len(df)}")

if len(df) == total_rows:
    print("Status: MANTAP! COCOK :)")
else:
    print("Status: ADA YANG MISSING / LEBIH :(")

print("\nCek kolom yang datanya kosong (NaN):")
print(df.isna().sum())

print("\nPreview 5 data pertama:")
display(df.head())

print(f"- {OUTPUT_XLSX}")
print(f"- {OUTPUT_CSV}")

print("\nOtw download file Excel-nya...")
files.download(OUTPUT_XLSX)


--- VALIDASI AKHIR ---
Target jumlah data dari web : 4001
Total yang berhasil discrap : 4001
Status: MANTAP! COCOK :)

Cek kolom yang datanya kosong (NaN):
Data source        0
Import & Export    0
date               0
HS code            0
Product DESC       0
Buyers             0
Supplier           0
Weight             0
Quantity           0
Amount             0
Purpose Country    0
Origin Country     0
dtype: int64

Preview 5 data pertama:


,Data source,Import & Export,date,HS code,Product DESC,Buyers,Supplier,Weight,Quantity,Amount,Purpose Country,Origin Country
0,Indonesia,Import,2026-08-28,02013000,DAGING SAPI SEGAR TANPA TULANG PRS RUMP,BAYU LESTARI,CAMPBELL FOODS AUSTRALIA PTY LTD,377.82,377.82,4439.8746,Indonesia,Australia
1,Indonesia,Import,2026-08-28,02013000,"DAGING SAPI SEGAR TANPA TULANG, A TENDERLOIN",BAYU LESTARI,CAMPBELL FOODS AUSTRALIA PTY LTD,141.91,141.91,4224.6539,Indonesia,Australia
2,Indonesia,Import,2026-08-28,02013000,"DAGING SAPI SEGAR TANPA TULANG, PRS CUBE ROLL",BAYU LESTARI,CAMPBELL FOODS AUSTRALIA PTY LTD,472.71,472.71,14813.2137,Indonesia,Australia
3,Indonesia,Import,2026-08-28,02013000,*YP-PEBO* POINT END BRISKET DECKLE OFF GRAIN FED,CLASSIC FINE FOODS INDONESIA,KILCOY PASTORAL COMPANY LIMITED,682.90,682.90,7107.4649,Indonesia,Australia
4,Indonesia,Import,2026-08-28,02013000,*YP-TDRSO* TENDERLOIN SIDE STRAP OFF GRAIN FED...,CLASSIC FINE FOODS INDONESIA,KILCOY PASTORAL COMPANY LIMITED,663.70,663.70,25010.4051,Indonesia,Australia


- /content/data1688_02013000_FINAL.xlsx
- /content/data1688_02013000_FINAL.csv

Otw download file Excel-nya...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>